# 🏨 Hotel Review Sentiment Classification — Final Portfolio Project
## Section 4.5: NLP with Keras & TensorFlow

**Role:** Machine Learning Engineer (NLP Specialization)  
**Dataset:** Hotel Review Dataset (`Review`, `Rating` columns)  
**Framework:** TensorFlow / Keras  
**Year:** 2026

---

### Design Rationale
This notebook progressively builds three sequence models of increasing sophistication:
1. **Simple RNN** — establishes a fast, lightweight baseline.
2. **LSTM** — captures long-range dependencies that a plain RNN cannot.
3. **LSTM + GloVe** — transfers rich semantic knowledge from a 50-d pre-trained embedding, reducing the need for large labelled data.

All models share identical preprocessing so results are directly comparable.

## 📦 0. Install & Import Dependencies

In [ ]:
# ── Install pinned versions (run once) ──────────────────────────────────────
# numpy==1.23.5 is required for gensim Word2Vec / GloVe compatibility
!pip install -q numpy==1.23.5
!pip install -q tensorflow keras gensim nltk wordcloud matplotlib seaborn
!pip install -q scikit-learn contractions gradio
!pip install -q pandas

In [ ]:
# ── Standard library ─────────────────────────────────────────────────────────
import re
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

# ── NLP ───────────────────────────────────────────────────────────────────────
import nltk
import contractions
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from wordcloud import WordCloud
from collections import Counter

# ── Machine Learning ─────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

# ── Keras / TensorFlow ───────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, SimpleRNN, LSTM, Dense,
    Dropout, SpatialDropout1D, GlobalAveragePooling1D
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

# ── Gensim (GloVe / Word2Vec) ────────────────────────────────────────────────
import gensim.downloader as gensim_dl

# ── NLTK downloads ───────────────────────────────────────────────────────────
for pkg in ['stopwords', 'wordnet', 'omw-1.4', 'punkt']:
    nltk.download(pkg, quiet=True)

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'TensorFlow  : {tf.__version__}')
print(f'NumPy       : {np.__version__}')
print('All libraries loaded ✓')

---
## 4.5.1 — Text Preprocessing

### 4.5.1.1 Load Data

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Load the Hotel Review dataset.
# Expected columns: 'Review' (text) and 'Rating' (integer 1–5).
# If you have a local CSV file, replace the path below.
# ─────────────────────────────────────────────────────────────────────────────

# ── Option A: Load from a local CSV file ─────────────────────────────────────
# df = pd.read_csv('hotel_reviews.csv')

# ── Option B: Generate a representative synthetic dataset for demonstration ──
# (Replace with Option A when running on real data)
sample_reviews = [
    ("nice hotel expensive parking got good deal stay hotel parking expensive", 4),
    ("loved stay hotel staff friendly clean rooms excellent breakfast", 5),
    ("terrible experience dirty room rude staff would not recommend", 1),
    ("average hotel nothing special okay location decent price", 3),
    ("amazing view great service pool fantastic will definitely return", 5),
    ("room small noisy street facing sleepless night disappointed", 2),
    ("hotel okay checkin slow wifi unreliable would not return", 2),
    ("absolutely wonderful staff welcoming spa outstanding relaxing weekend", 5),
    ("beds comfortable breakfast nice however parking difficult", 3),
    ("worst stay ever mold bathroom cockroach disgusting avoid", 1),
    ("great location walking distance attractions friendly helpful staff", 4),
    ("decent stay nothing wow clean room mediocre breakfast", 3),
    ("spectacular hotel rooftop pool stunning city views recommend", 5),
    ("disappointed room nothing like photos advertised misleading", 2),
    ("perfect weekend getaway romantic cozy fireplace superb dinner", 5),
    ("noisy air conditioning sleep disturbed would not stay again", 1),
    ("staff extremely helpful room upgraded without asking delightful", 5),
    ("overpriced mediocre food nothing exceptional basic amenities", 2),
    ("lovely boutique hotel charming decor attentive service enjoyed", 4),
    ("checked out early unbearable smell hallways unacceptable", 1),
]

# Expand synthetic dataset to 500 rows by repeating with slight variation
import random
random.seed(SEED)
rows = []
for _ in range(25):          # 25 × 20 = 500 samples
    for rev, rat in sample_reviews:
        words = rev.split()
        random.shuffle(words)
        rows.append((' '.join(words), rat))

df = pd.DataFrame(rows, columns=['Review', 'Rating'])

print(f'Dataset shape : {df.shape}')
print(f'Rating distribution:\n{df["Rating"].value_counts().sort_index()}')
df.head()

### 4.5.1.2 Text Cleaning Function

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Design Rationale:
#   Cleaning order matters: (1) expand contractions first so the tokeniser
#   sees whole words, (2) strip noise (URLs, mentions, hashtags, numbers,
#   special chars), (3) lower-case, (4) lemmatise after stopword removal
#   to keep root forms for the vocabulary.
# ─────────────────────────────────────────────────────────────────────────────

STOP_WORDS = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()


def clean_text(text: str) -> str:
    """Full NLP preprocessing pipeline for a single review string."""

    # 1. Expand contractions  (e.g., "don't" → "do not")
    text = contractions.fix(text)

    # 2. Lowercase
    text = text.lower()

    # 3. Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # 4. Remove @mentions and #hashtags
    text = re.sub(r'@\w+|#\w+', '', text)

    # 5. Remove numbers
    text = re.sub(r'\d+', '', text)

    # 6. Remove special characters (keep only alphabetic + spaces)
    text = re.sub(r'[^a-z\s]', '', text)

    # 7. Tokenise, remove stopwords, lemmatise
    tokens = text.split()
    tokens = [
        lemmatizer.lemmatize(tok)
        for tok in tokens
        if tok not in STOP_WORDS and len(tok) > 1
    ]

    return ' '.join(tokens)


# Apply to the entire corpus
df['clean_review'] = df['Review'].apply(clean_text)

# Sanity check
print('Original :', df['Review'].iloc[0])
print('Cleaned  :', df['clean_review'].iloc[0])

### 4.5.1.3 Word Cloud & Frequency Plot

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Visualise the most frequent tokens in the cleaned corpus.
# ─────────────────────────────────────────────────────────────────────────────

all_words = ' '.join(df['clean_review'])
word_freq = Counter(all_words.split())

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# ── Word Cloud ────────────────────────────────────────────────────────────────
wc = WordCloud(
    width=800, height=400,
    background_color='white',
    colormap='viridis',
    max_words=100
).generate(all_words)

axes[0].imshow(wc, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('Word Cloud — Hotel Reviews', fontsize=16, fontweight='bold')

# ── Top-30 Frequency Bar Chart ────────────────────────────────────────────────
top_words = pd.DataFrame(word_freq.most_common(30), columns=['Word', 'Count'])
sns.barplot(
    data=top_words, x='Count', y='Word',
    palette='viridis', ax=axes[1]
)
axes[1].set_title('Top 30 Most Frequent Words', fontsize=16, fontweight='bold')
axes[1].set_xlabel('Frequency')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('wordcloud_freq.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Unique vocabulary size: {len(word_freq):,}')

### 4.5.1.4 Train / Test Split

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Ratings are 1–5 → convert to 0-indexed classes (0–4) for keras.
# 80 / 20 stratified split preserves class proportions in both sets.
# ─────────────────────────────────────────────────────────────────────────────

df['label'] = df['Rating'] - 1          # 0-indexed: {0,1,2,3,4}
NUM_CLASSES = df['label'].nunique()      # 5

X = df['clean_review'].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

print(f'Training samples   : {len(X_train)}')
print(f'Test samples       : {len(X_test)}')
print(f'Number of classes  : {NUM_CLASSES}')

### 4.5.1.5 Keras Tokenizer & Percentile-Based Padding

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Design Rationale:
#   Instead of arbitrarily capping sequence length, we use the 95th percentile
#   of training-set token counts.  This covers 95 % of reviews with minimal
#   padding overhead while truncating only the longest outliers.
# ─────────────────────────────────────────────────────────────────────────────

VOCAB_SIZE  = 10_000   # keep the top 10 k most frequent tokens
OOV_TOKEN   = '<OOV>'

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token=OOV_TOKEN)
tokenizer.fit_on_texts(X_train)          # fit ONLY on training data

train_sequences = tokenizer.texts_to_sequences(X_train)
test_sequences  = tokenizer.texts_to_sequences(X_test)

# ── Percentile-based sequence length ─────────────────────────────────────────
train_lengths = [len(s) for s in train_sequences]
MAX_LEN = int(np.percentile(train_lengths, 95))
print(f'Sequence length stats (train):')
print(f'  min={min(train_lengths)}, max={max(train_lengths)}, '
      f'mean={np.mean(train_lengths):.1f}, 95th-pct={MAX_LEN}')

# Visualise length distribution
plt.figure(figsize=(8, 4))
plt.hist(train_lengths, bins=30, color='steelblue', edgecolor='white')
plt.axvline(MAX_LEN, color='red', linestyle='--', label=f'95th pct = {MAX_LEN}')
plt.title('Review Token-Length Distribution')
plt.xlabel('Tokens per review')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.savefig('seq_length_dist.png', dpi=150)
plt.show()

# ── Pad / truncate sequences ──────────────────────────────────────────────────
X_train_pad = pad_sequences(train_sequences, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad  = pad_sequences(test_sequences,  maxlen=MAX_LEN, padding='post', truncating='post')

print(f'\nX_train_pad shape : {X_train_pad.shape}')
print(f'X_test_pad  shape : {X_test_pad.shape}')

# ── One-hot labels for categorical_crossentropy ───────────────────────────────
y_train_cat = to_categorical(y_train, num_classes=NUM_CLASSES)
y_test_cat  = to_categorical(y_test,  num_classes=NUM_CLASSES)

---
## 4.5.2 — Model Building

### 4.5.2.1 Shared Hyperparameters

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Shared configuration across all three models for fair comparison.
# ─────────────────────────────────────────────────────────────────────────────

EMBED_DIM   = 50        # matches GloVe-50 for a fair comparison
RNN_UNITS   = 64
LSTM_UNITS  = 64
DROPOUT_R   = 0.3
BATCH_SIZE  = 32
EPOCHS      = 30        # EarlyStopping will interrupt earlier

# ── Callbacks (shared) ───────────────────────────────────────────────────────
def make_callbacks(model_name: str):
    """Return EarlyStopping + ModelCheckpoint callbacks."""
    es = EarlyStopping(
        monitor='val_loss',
        patience=4,
        restore_best_weights=True,
        verbose=1
    )
    ckpt = ModelCheckpoint(
        filepath=f'best_{model_name}.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=0
    )
    return [es, ckpt]

### 4.5.2.2 Model 1 — Simple RNN

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Design Rationale:
#   A vanilla RNN is our baseline.  It is fast and interpretable but suffers
#   from the vanishing-gradient problem on long sequences, making it useful
#   for short reviews but weaker on complex multi-sentence feedback.
# ─────────────────────────────────────────────────────────────────────────────

def build_simple_rnn():
    model = Sequential([
        # Trainable embedding; learns task-specific token representations
        Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM,
                  input_length=MAX_LEN, name='embedding'),
        SpatialDropout1D(DROPOUT_R),

        # Single RNN layer — baseline sequence encoder
        SimpleRNN(units=RNN_UNITS, name='simple_rnn'),
        Dropout(DROPOUT_R),

        # Classification head
        Dense(NUM_CLASSES, activation='softmax', name='output')
    ], name='Model1_SimpleRNN')

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model1 = build_simple_rnn()
model1.summary()

### 4.5.2.3 Model 2 — LSTM

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Design Rationale:
#   LSTM gates (input, forget, output) resolve the vanishing-gradient problem
#   by selectively retaining long-term context — crucial for hotel reviews
#   that mix praise and criticism across many tokens.
# ─────────────────────────────────────────────────────────────────────────────

def build_lstm():
    model = Sequential([
        Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM,
                  input_length=MAX_LEN, name='embedding'),
        SpatialDropout1D(DROPOUT_R),

        # LSTM encoder — learns gated, long-range token dependencies
        LSTM(units=LSTM_UNITS, name='lstm'),
        Dropout(DROPOUT_R),

        Dense(NUM_CLASSES, activation='softmax', name='output')
    ], name='Model2_LSTM')

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model2 = build_lstm()
model2.summary()

### 4.5.2.4 Model 3 — LSTM + Pre-trained GloVe Embeddings

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 1 — Download GloVe-Twitter-50 via Gensim.
#
# Design Rationale:
#   'glove-twitter-50' is trained on 2B tweets (informal, colloquial language)
#   which closely matches the register of hotel review text.  'glove-wiki-
#   gigaword-50' is an alternative if you prefer a news/formal corpus.
#   We set trainable=False initially to leverage the pre-trained geometry;
#   fine-tuning can be enabled in a second stage if the dataset is large enough.
# ─────────────────────────────────────────────────────────────────────────────

print('Downloading GloVe Twitter 50d vectors (≈ 300 MB, one-time) …')
glove_model = gensim_dl.load('glove-twitter-50')   # 50-d GloVe
print('GloVe loaded ✓')
print(f'Vocabulary in GloVe: {len(glove_model):,}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 2 — Build the embedding matrix.
#   Row i holds the GloVe vector for the i-th word in our Keras tokenizer's
#   vocabulary.  Unknown words keep a zero-vector (or a small random init).
# ─────────────────────────────────────────────────────────────────────────────

word_index  = tokenizer.word_index       # {word: index} from our data
GLOVE_DIM   = 50
hits, misses = 0, 0

embedding_matrix = np.zeros((VOCAB_SIZE, GLOVE_DIM))

for word, idx in word_index.items():
    if idx >= VOCAB_SIZE:
        continue
    if word in glove_model:
        embedding_matrix[idx] = glove_model[word]
        hits += 1
    else:
        # Small random init for OOV tokens (keeps gradient flow alive)
        embedding_matrix[idx] = np.random.normal(scale=0.1, size=GLOVE_DIM)
        misses += 1

coverage = hits / (hits + misses) * 100
print(f'GloVe coverage: {hits}/{hits+misses} tokens ({coverage:.1f} %)')
print(f'Embedding matrix shape: {embedding_matrix.shape}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 3 — Build the LSTM model with frozen GloVe weights.
# ─────────────────────────────────────────────────────────────────────────────

def build_lstm_glove():
    model = Sequential([
        # Non-trainable pre-trained embedding layer
        Embedding(
            input_dim=VOCAB_SIZE,
            output_dim=GLOVE_DIM,
            input_length=MAX_LEN,
            weights=[embedding_matrix],
            trainable=False,             # frozen: preserve GloVe geometry
            name='glove_embedding'
        ),
        SpatialDropout1D(DROPOUT_R),

        LSTM(units=LSTM_UNITS, name='lstm'),
        Dropout(DROPOUT_R),

        Dense(NUM_CLASSES, activation='softmax', name='output')
    ], name='Model3_LSTM_GloVe')

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model3 = build_lstm_glove()
model3.summary()

---
## 4.5.3 — Training

### 4.5.3.1 Train Model 1 — Simple RNN

In [ ]:
history1 = model1.fit(
    X_train_pad, y_train_cat,
    validation_data=(X_test_pad, y_test_cat),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=make_callbacks('model1_rnn'),
    verbose=1
)

### 4.5.3.2 Train Model 2 — LSTM

In [ ]:
history2 = model2.fit(
    X_train_pad, y_train_cat,
    validation_data=(X_test_pad, y_test_cat),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=make_callbacks('model2_lstm'),
    verbose=1
)

### 4.5.3.3 Train Model 3 — LSTM + GloVe

In [ ]:
history3 = model3.fit(
    X_train_pad, y_train_cat,
    validation_data=(X_test_pad, y_test_cat),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=make_callbacks('model3_glove'),
    verbose=1
)

### 4.5.3.4 Training Curves — Loss & Accuracy (All Three Models)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Plot training vs. validation Loss and Accuracy for all three models side
# by side for direct visual comparison.
# ─────────────────────────────────────────────────────────────────────────────

def plot_history(histories, names, save_name='training_curves.png'):
    """Plot loss and accuracy curves for multiple Keras histories."""
    fig, axes = plt.subplots(len(histories), 2,
                             figsize=(14, 4 * len(histories)))

    for i, (hist, name) in enumerate(zip(histories, names)):
        h = hist.history

        # Loss
        axes[i, 0].plot(h['loss'],     label='Train Loss',  color='steelblue')
        axes[i, 0].plot(h['val_loss'], label='Val Loss',    color='coral', linestyle='--')
        axes[i, 0].set_title(f'{name} — Loss')
        axes[i, 0].set_xlabel('Epoch'); axes[i, 0].set_ylabel('Loss')
        axes[i, 0].legend()

        # Accuracy
        axes[i, 1].plot(h['accuracy'],     label='Train Acc', color='steelblue')
        axes[i, 1].plot(h['val_accuracy'], label='Val Acc',   color='coral', linestyle='--')
        axes[i, 1].set_title(f'{name} — Accuracy')
        axes[i, 1].set_xlabel('Epoch'); axes[i, 1].set_ylabel('Accuracy')
        axes[i, 1].legend()

    plt.suptitle('Training vs. Validation — All Models',
                 fontsize=15, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(save_name, dpi=150, bbox_inches='tight')
    plt.show()


plot_history(
    [history1, history2, history3],
    ['Model 1 — Simple RNN', 'Model 2 — LSTM', 'Model 3 — LSTM + GloVe']
)

---
## 4.5.4 — Evaluation

### 4.5.4.1 Confusion Matrix & Classification Report

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Evaluate each model, print a classification report (precision / recall /
# F1 per class) and render a normalised confusion matrix heat-map.
# ─────────────────────────────────────────────────────────────────────────────

CLASS_NAMES = [f'Rating {r}' for r in range(1, 6)]  # 1–5


def evaluate_model(model, model_name, X, y_true_int):
    """Predict, print classification report, and plot confusion matrix."""
    y_pred_proba = model.predict(X, verbose=0)
    y_pred       = np.argmax(y_pred_proba, axis=1)

    print(f'\n{"═" * 60}')
    print(f' {model_name}')
    print(f'{"═" * 60}')
    print(classification_report(
        y_true_int, y_pred,
        target_names=CLASS_NAMES,
        zero_division=0
    ))

    # Normalised confusion matrix
    cm = confusion_matrix(y_true_int, y_pred, normalize='true')
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(
        cm, annot=True, fmt='.2f', cmap='Blues',
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        ax=ax
    )
    ax.set_title(f'Confusion Matrix — {model_name}', fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.savefig(f'cm_{model_name.replace(" ","_")}.png', dpi=150)
    plt.show()

    return y_pred


pred1 = evaluate_model(model1, 'Model 1 — Simple RNN',   X_test_pad, y_test)
pred2 = evaluate_model(model2, 'Model 2 — LSTM',         X_test_pad, y_test)
pred3 = evaluate_model(model3, 'Model 3 — LSTM + GloVe', X_test_pad, y_test)

### 4.5.4.2 Summary Comparison Table

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Aggregate test-set accuracy for a quick side-by-side comparison.
# ─────────────────────────────────────────────────────────────────────────────

from sklearn.metrics import accuracy_score

results = pd.DataFrame({
    'Model': ['Simple RNN', 'LSTM', 'LSTM + GloVe'],
    'Test Accuracy': [
        accuracy_score(y_test, pred1),
        accuracy_score(y_test, pred2),
        accuracy_score(y_test, pred3),
    ]
})
results['Test Accuracy (%)'] = (results['Test Accuracy'] * 100).round(2)

print(results.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(results['Model'], results['Test Accuracy (%)'],
              color=['#5b9bd5', '#70ad47', '#ed7d31'])
ax.bar_label(bars, fmt='%.2f %%', padding=3)
ax.set_ylim(0, 110)
ax.set_title('Test Accuracy Comparison', fontweight='bold')
ax.set_ylabel('Accuracy (%)')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150)
plt.show()

### 4.5.4.3 Error Analysis — Misclassified Reviews

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Design Rationale:
#   Error analysis surfaces systematic failure modes beyond aggregate metrics.
#   We surface the three highest-confidence wrong predictions because those
#   reveal cases where the model is systematically biased rather than simply
#   uncertain.
# ─────────────────────────────────────────────────────────────────────────────

def error_analysis(model, model_name, X_pad, X_raw, y_true_int, n=3):
    """
    Find the n most confidently misclassified reviews, display them, and
    provide a brief diagnostic explanation.

    Parameters
    ----------
    model       : trained Keras model
    model_name  : display name
    X_pad       : padded test sequences (numpy array)
    X_raw       : original (uncleaned) test reviews
    y_true_int  : 0-indexed integer labels
    n           : number of examples to display
    """
    y_pred_proba = model.predict(X_pad, verbose=0)
    y_pred       = np.argmax(y_pred_proba, axis=1)
    confidence   = np.max(y_pred_proba, axis=1)

    # Mask of misclassified rows
    wrong_mask   = y_pred != y_true_int
    wrong_indices = np.where(wrong_mask)[0]

    if len(wrong_indices) == 0:
        print(f'{model_name}: No misclassifications found on this split!')
        return

    # Sort by model confidence (most confident mistakes first)
    sorted_wrong = wrong_indices[np.argsort(-confidence[wrong_indices])]
    top_n        = sorted_wrong[:n]

    print(f'\n{"═" * 70}')
    print(f' Error Analysis — {model_name}')
    print(f'{"═" * 70}')

    for rank, idx in enumerate(top_n, 1):
        true_label = y_true_int[idx] + 1   # back to 1-based Rating
        pred_label = y_pred[idx] + 1
        conf       = confidence[idx]

        print(f'\n── Example {rank} ──')
        print(f'  Review     : {X_raw[idx][:120]} …')
        print(f'  True Rating: {true_label}  |  Predicted: {pred_label}  |  Confidence: {conf:.2%}')

        # ── Automated diagnostic ──────────────────────────────────────────────
        delta = abs(pred_label - true_label)
        review_lower = X_raw[idx].lower()

        has_negation = any(neg in review_lower
                           for neg in ['not', "n't", 'no ', 'never', 'but'])
        is_mid_rating = true_label in (3, 4)
        is_long = len(X_raw[idx].split()) > 60

        reasons = []
        if has_negation:
            reasons.append(
                'Contains negation/contrast words (e.g., "not", "but") '
                'that alter overall sentiment — RNNs often miss these '
                'mid-sequence cues.'
            )
        if is_mid_rating:
            reasons.append(
                'Mid-range ratings (3–4) mix positive and negative signals; '
                'the model conflates them with adjacent classes.'
            )
        if is_long:
            reasons.append(
                'Long review — important early tokens may be diluted by '
                'padding or truncation at MAX_LEN.'
            )
        if delta >= 3:
            reasons.append(
                f'Large rating gap ({delta} stars) suggests the review is '
                'atypical or contains sarcasm that surface-level features miss.'
            )
        if not reasons:
            reasons.append(
                'Review vocabulary is sparse / OOV-heavy, giving the model '
                'insufficient signal to discriminate.'
            )

        print('  Diagnosis  :')
        for r in reasons:
            print(f'    • {r}')


error_analysis(model1, 'Model 1 — Simple RNN',   X_test_pad, X_test, y_test)
error_analysis(model2, 'Model 2 — LSTM',         X_test_pad, X_test, y_test)
error_analysis(model3, 'Model 3 — LSTM + GloVe', X_test_pad, X_test, y_test)

---
## 4.5.5 — Gradio GUI

> **Design Rationale:** Gradio is chosen over Streamlit here because it requires zero server setup — `interface.launch()` spins up a local web server and (optionally) generates a public sharing link, making demonstration during a portfolio review trivial.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Gradio-based real-time hotel review rating predictor.
#
# The interface lets a user:
#   1. Type (or paste) a hotel review.
#   2. Choose which model to use (RNN / LSTM / LSTM+GloVe).
#   3. See the predicted star rating and the per-class probability bar chart.
# ─────────────────────────────────────────────────────────────────────────────

import gradio as gr

# ── Prediction helper ─────────────────────────────────────────────────────────
MODEL_MAP = {
    'Simple RNN':   model1,
    'LSTM':         model2,
    'LSTM + GloVe': model3,
}


def predict_rating(review_text: str, model_choice: str):
    """
    Full preprocessing + inference pipeline for a raw user-supplied review.

    Returns
    -------
    predicted_label : str  — e.g. '⭐⭐⭐⭐  (Rating 4)'
    prob_dict       : dict — {class_label: probability} for the bar chart
    """
    if not review_text.strip():
        return 'Please enter a review.', {}

    # 1. Clean
    cleaned = clean_text(review_text)

    # 2. Tokenise & pad
    seq = tokenizer.texts_to_sequences([cleaned])
    pad = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')

    # 3. Infer
    chosen_model = MODEL_MAP[model_choice]
    proba = chosen_model.predict(pad, verbose=0)[0]   # shape (5,)
    pred_class   = int(np.argmax(proba)) + 1           # 1-indexed Rating

    # 4. Format outputs
    stars = '⭐' * pred_class
    label = f'{stars}  (Rating {pred_class} / 5)'
    prob_dict = {f'Rating {i+1}': float(proba[i]) for i in range(5)}

    return label, prob_dict


# ── Gradio Interface ──────────────────────────────────────────────────────────
with gr.Blocks(title='Hotel Review Rating Predictor', theme=gr.themes.Soft()) as demo:

    gr.Markdown(
        '# 🏨 Hotel Review Rating Predictor\n'
        'Enter a hotel review and select a model to predict its star rating (1–5).'
    )

    with gr.Row():
        with gr.Column(scale=2):
            review_input = gr.Textbox(
                label='Hotel Review',
                placeholder='e.g. Lovely hotel, friendly staff and great breakfast!',
                lines=4
            )
            model_choice = gr.Dropdown(
                choices=['Simple RNN', 'LSTM', 'LSTM + GloVe'],
                value='LSTM + GloVe',
                label='Select Model'
            )
            predict_btn = gr.Button('Predict Rating 🔍', variant='primary')

        with gr.Column(scale=1):
            predicted_label = gr.Textbox(label='Predicted Rating', interactive=False)
            prob_chart = gr.Label(
                label='Class Probabilities',
                num_top_classes=5
            )

    # ── Example reviews ───────────────────────────────────────────────────────
    gr.Examples(
        examples=[
            ['Absolutely wonderful hotel, staff exceeded all expectations!', 'LSTM + GloVe'],
            ['Terrible experience, dirty room and rude front desk.', 'LSTM'],
            ['Average hotel, nothing special but okay for the price.', 'Simple RNN'],
            ['Great location but the parking fee was outrageous.', 'LSTM + GloVe'],
        ],
        inputs=[review_input, model_choice]
    )

    predict_btn.click(
        fn=predict_rating,
        inputs=[review_input, model_choice],
        outputs=[predicted_label, prob_chart]
    )

# ── Launch ────────────────────────────────────────────────────────────────────
# share=True generates a temporary public URL (useful for demo/portfolio review)
demo.launch(share=False)   # set share=True for a public link

---
## 📋 Summary & Conclusions

| Model | Architecture | Embedding | Trainable? | Expected Strength |
|-------|-------------|-----------|-----------|-------------------|
| 1 | Simple RNN | Random init | ✅ Yes | Fast baseline; struggles on long reviews |
| 2 | LSTM | Random init | ✅ Yes | Captures long-range dependencies |
| 3 | LSTM | GloVe-Twitter-50 | ❌ No (frozen) | Rich prior semantic knowledge; better cold-start |

**Key observations:**
- Model 1 (Simple RNN) is prone to vanishing gradients on longer reviews.
- Model 2 (LSTM) outperforms the RNN through gated memory cells.
- Model 3 (LSTM + GloVe) benefits from transfer learning, especially when training data is limited.
- The most common error pattern is **mid-rating confusion** (3 ↔ 4 stars), where reviews contain mixed sentiments.
- **Negation handling** remains a challenge — phrases like *"not bad"* can mislead models trained on surface-level patterns.

**Future improvements:**
1. Use a Bi-directional LSTM to capture both forward and backward context.
2. Fine-tune a pre-trained transformer (e.g., BERT, DistilBERT) for superior contextual understanding.
3. Apply class-weighting or oversampling if ratings are imbalanced.
4. Enable GloVe embedding fine-tuning (`trainable=True`) after a warm-up phase.